# 7. Modelos:  BPE, Embeddings, Neural LM

<a target="_blank" href="https://colab.research.google.com/github/umoqnier/cl-2026-2-lab/blob/main/notebooks/7_neural_lm.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Objetivos

- Entrenar modelos para sub-word tokenization
  - Aplicar BPE a corpus
- Entrenar modelos para *embeddings*
  - Word2Vec
  - Glove
- Implementación de modelo del lenguaje Neuronal de Bengio
  - Generación de lenguaje

In [1]:
import os
import re
from rich import print as rprint
from nltk import word_tokenize
from collections import Counter
from nltk.stem.snowball import SnowballStemmer

## Funciones de preprocesamiento

In [ ]:
from nltk.corpus import stopwords
import unicodedata


def strip_accents(s: str) -> str:
    """Remove diacritical marks from characters in a Unicode string.

    Uses Unicode NFD (Normalization Form Decomposition) normalization to decompose accented characters into their
    base character + combining mark, then filters out combining marks (Mark, Nonspacing, Mn category).
    """
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )


def preprocess_words(
    words: list[str],
    regex: str = r"[^\w+]",
    lang: str = "en",
    remove_stops: bool = False,
    remove_accents: bool = False,
) -> list[str]:
    """Preprocess step for list of words in corpus"""
    stop_lang = "english" if lang == "en" else "spanish"
    result = []
    for word in words:
        word = re.sub(regex, "", word).lower()
        if remove_stops and word in stopwords.words(stop_lang) or (len(word) < 2):
            continue

        if remove_accents:
            word = strip_accents(word)
        result.append(word)
    return result


def preprocess_text(text: str, to_lower: bool = True) -> str:
    # 1. Unicode Normalization (NFC)
    text = unicodedata.normalize("NFC", text)

    if to_lower:
        text = text.lower()

    # 3. Collapse all whitespace/newlines into a single space
    text = re.sub(r"\s+", " ", text)

    # 4. Clean up leading/trailing whitespace
    text = text.strip()

    return text

## Byte-Pair Encoding

### Vamos a tokenizar 🌈
![](https://i.pinimg.com/736x/58/6b/88/586b8825f010ce0e3f9c831f568aafa8.jpg)

In [4]:
BASE_PATH = "."
CORPORA_PATH = f"{BASE_PATH}/data/"
MODELS_PATH = f"{BASE_PATH}/models/"

os.makedirs(CORPORA_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)


In [5]:
TOKENIZERS_DATA_PATH = f"{MODELS_PATH}/tokenization"
TOKENIZERS_MODEL_PATH = f"{TOKENIZERS_DATA_PATH}/sub-word"

os.makedirs(TOKENIZERS_DATA_PATH, exist_ok=True)
os.makedirs(TOKENIZERS_MODEL_PATH, exist_ok=True)

### Corpus en español: Wikipedia

In [6]:
from datasets import load_dataset, load_dataset_builder

In [6]:
data_builder = load_dataset_builder("wikimedia/wikipedia", "20231101.es")

In [7]:
rprint(data_builder.info)

DatasetInfo(
    description='',
    citation='',
    homepage='',
    license='',
    features={
        'id': Value(dtype='string', id=None),
        'url': Value(dtype='string', id=None),
        'title': Value(dtype='string', id=None),
        'text': Value(dtype='string', id=None)
    },
    post_processed=None,
    supervised_keys=None,
    builder_name='parquet',
    dataset_name='wikipedia',
    config_name='20231101.es',
    version=0.0.0,
    splits={
        'train': SplitInfo(
            name='train',
            num_bytes=6033536133,
            num_examples=1841155,
            shard_lengths=None,
            dataset_name=None
        )
    },
    download_checksums=None,
    download_size=3493595869,
    post_processing_size=None,
    dataset_size=6033536133,
    size_in_bytes=None
)

In [ ]:
dataset = load_dataset(
    "wikimedia/wikipedia", "20231101.es", split="train", streaming=True
)

In [9]:
wiki_words = []
for article in dataset.take(1):
    rprint(preprocess_text(article["text"][:1000], to_lower=False))

Andorra, oficialmente Principado de Andorra () es un micro-Estado soberano sin litoral ubicado en el suroeste de 
Europa, entre España y Francia, en el límite de la península ibérica. Se constituye en Estado independiente, de 
derecho, democrático y social, cuya forma de gobierno es el coprincipado parlamentario. Su territorio está 
organizado en siete parroquias, con una población total de 79 877 habitantes a 28 de febrero de 2022. Su capital es
Andorra la Vieja. Con sus 468 km² de extensión territorial, Andorra es el micro-Estado más grande de Europa y está 
situado en los Pirineos, entre España y Francia; tiene una altitud media de 1996ms.n.m. Limita por el sur con 
España —con la provincia catalana de Lérida— y por el norte con Francia —con los departamentos de Ariège y Pirineos
Orientales (Occitania)—. Pertenece culturalmente a la Europa latina. Su sistema político es una democracia 
parlamentaria cuyos jefes de Estado son los copríncipes de Andorra: el obispo de Urgel y el presidente

In [12]:
%%time

wiki_file_path = f"{CORPORA_PATH}/wikipedia_es_plain.txt"
with open(wiki_file_path, "w", encoding="utf-8") as f:
    for article in dataset.take(1000):
        f.write(preprocess_text(article["text"]))
        f.write("\n")

CPU times: user 1.01 s, sys: 172 ms, total: 1.19 s
Wall time: 5.68 s


In [14]:
!head -n 10 {CORPORA_PATH}/wikipedia_es_plain.txt

andorra, oficialmente principado de andorra () es un micro-estado soberano sin litoral ubicado en el suroeste de europa, entre españa y francia, en el límite de la península ibérica. se constituye en estado independiente, de derecho, democrático y social, cuya forma de gobierno es el coprincipado parlamentario. su territorio está organizado en siete parroquias, con una población total de 79 877 habitantes a 28 de febrero de 2022. su capital es andorra la vieja. con sus 468 km² de extensión territorial, andorra es el micro-estado más grande de europa y está situado en los pirineos, entre españa y francia; tiene una altitud media de 1996ms.n.m. limita por el sur con españa —con la provincia catalana de lérida— y por el norte con francia —con los departamentos de ariège y pirineos orientales (occitania)—. pertenece culturalmente a la europa latina. su sistema político es una democracia parlamentaria cuyos jefes de estado son los copríncipes de andorra: el obispo de urgel y el presidente d

### Entrenando nuestro modelo con BPE

![](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmedia1.tenor.com%2Fimages%2Fd565618bb1217a7c435579d9172270d0%2Ftenor.gif%3Fitemid%3D3379322&f=1&nofb=1&ipt=9719714edb643995ce9d978c8bab77f5310204960093070e37e183d5372096d9&ipo=images)

In [ ]:
!pip install subword-nmt

In [15]:
!ls {CORPORA_PATH}

cess_plain.txt		 gutenberg_plain.txt	  wikipedia_es_plain.txt
eng-bible-tokenized.txt  spa-bible-tokenized.txt
eng-bible.txt		 spa-bible.txt


In [18]:
!subword-nmt --help

usage: subword-nmt [-h]
                   {learn-bpe,apply-bpe,get-vocab,learn-joint-bpe-and-vocab}
                   ...

subword-nmt: unsupervised word segmentation for neural machine translation and text generation 

positional arguments:
  {learn-bpe,apply-bpe,get-vocab,learn-joint-bpe-and-vocab}
                        command to run. Run one of the commands with '-h' for more info.
                        
                        learn-bpe: learn BPE merge operations on input text.
                        apply-bpe: apply given BPE operations to input text.
                        get-vocab: extract vocabulary and word frequencies from input text.
                        learn-joint-bpe-and-vocab: executes recommended workflow for joint BPE.

options:
  -h, --help            show this help message and exit


In [19]:
!subword-nmt learn-bpe --help

usage: subword-nmt learn-bpe [-h] [--input PATH] [--output PATH]
                             [--symbols SYMBOLS] [--min-frequency FREQ]
                             [--dict-input] [--total-symbols]
                             [--num-workers NUM_WORKERS] [--verbose]

learn BPE-based word segmentation

options:
  -h, --help            show this help message and exit
  --input PATH, -i PATH
                        Input text (default: standard input).
  --output PATH, -o PATH
                        Output file for BPE codes (default: standard output)
  --symbols SYMBOLS, -s SYMBOLS
                        Create this many new symbols (each representing a
                        character n-gram) (default: 10000)
  --min-frequency FREQ  Stop if no symbol pair has frequency >= FREQ (default:
                        2)
  --dict-input          If set, input file is interpreted as a dictionary
                        where each line contains a word-count pair
  --total-symbols, -t   subtrac

In [21]:
%%time

!subword-nmt learn-bpe --num-workers -1 -s 300 < \
 {CORPORA_PATH}/wikipedia_es_plain.txt > \
  {MODELS_PATH}/wiki_es_300.model

100%|#########################################| 300/300 [00:07<00:00, 42.42it/s]
CPU times: user 169 ms, sys: 51.9 ms, total: 221 ms
Wall time: 10.5 s


In [23]:
!echo "ando haciendo un análisis para claramente ver si puedes procesar esta oración mano" \
| subword-nmt apply-bpe -c {MODELS_PATH}/wiki_es_300.model

ando h@@ aci@@ en@@ do un an@@ á@@ l@@ is@@ i@@ s para cl@@ ar@@ amente v@@ er s@@ i pu@@ ed@@ es pro@@ c@@ es@@ ar esta or@@ ación man@@ o


In [24]:
%%time

!subword-nmt learn-bpe --num-workers -1 -s 10000 < \
data/tokenization/wikipedia_es_plain.txt > \
 models/sub-word/wiki_es_10k.model

100%|####################################| 10000/10000 [00:25<00:00, 397.88it/s]
CPU times: user 418 ms, sys: 104 ms, total: 522 ms
Wall time: 28.5 s


In [26]:
!echo "ando haciendo un análisis para claramente ver si puedes procesar esta oración mano" \
| subword-nmt apply-bpe -c models/sub-word/wiki_es_10k.model

ando haciendo un análisis para claramente ver si pued@@ es proces@@ ar esta or@@ ación mano


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(
    " ".join(
        tokenizer.tokenize(
            "ando haciendo un análisis para claramente ver si puedes procesar esta oración mano"
        )
    ).replace("Ġ", "@@")
)

ando @@h aci endo @@un @@an Ã¡ lis is @@para @@clar ament e @@ver @@si @@p ued es @@pro ces ar @@est a @@or aci Ã³n @@man o


### Aplicandolo a otros corpus: La biblia 📖🇻🇦

In [39]:
BIBLE_FILE_NAMES = {
    "spa": "spa-x-bible-reinavaleracontemporanea",
    "eng": "eng-x-bible-kingjames",
}

In [40]:
import requests


def get_bible_corpus(lang: str) -> str:
    """Download bible file corpus from GitHub repo"""
    file_name = BIBLE_FILE_NAMES[lang]
    r = requests.get(
        f"https://raw.githubusercontent.com/ximenina/theturningpoint/main/Detailed/corpora/corpusPBC/{file_name}.txt.clean.txt"
    )
    return r.text


def write_plain_text_corpus(raw_text: str, file_name: str) -> None:
    """Write file text on disk"""
    with open(f"{file_name}.txt", "w") as f:
        f.write(raw_text)

#### Tokenizando biblia en español

In [53]:
spa_bible_raw = get_bible_corpus("spa")
spa_bible_plain_text = preprocess_text(spa_bible_raw)

In [54]:
write_plain_text_corpus(spa_bible_plain_text, f"{CORPORA_PATH}/bible-spa")

In [55]:
!subword-nmt apply-bpe -c {MODELS_PATH}/wiki_es_10k.model < \
 {CORPORA_PATH}/bible-spa.txt > \
 {CORPORA_PATH}/bible-spa-tokenized.txt

#### Comparando resultados

In [ ]:
spa_bible_words = word_tokenize(spa_bible_plain_text)

In [58]:
spa_bible_words[:10]

['principio',
 'del',
 'evangelio',
 'de',
 'jesucristo',
 ',',
 'el',
 'hijo',
 'de',
 'dios']

In [59]:
len(spa_bible_words)

30073

In [62]:
spa_bible_types = Counter(spa_bible_words)
len(spa_bible_types)

3317

In [63]:
spa_bible_types.most_common(30)

[(',', 1946),
 ('y', 1169),
 ('.', 1099),
 ('de', 1009),
 ('que', 927),
 ('a', 858),
 ('los', 645),
 ('la', 599),
 ('el', 572),
 (':', 539),
 ('se', 489),
 ('en', 461),
 ('«', 423),
 ('»', 423),
 ('jesús', 422),
 ('lo', 367),
 ('no', 312),
 ('le', 293),
 ('les', 267),
 ('dijo', 252),
 ('con', 220),
 ('pero', 217),
 ('al', 214),
 ('¿', 196),
 ('?', 195),
 ('por', 194),
 ('para', 172),
 ('su', 171),
 ('del', 165),
 ('un', 159)]

In [64]:
with open(f"{CORPORA_PATH}/bible-spa-tokenized.txt", "r") as f:
    tokenized_text = f.read()
spa_bible_tokenized = tokenized_text.split()

In [65]:
spa_bible_tokenized[:10]

['principio',
 'del',
 'evangelio',
 'de',
 'jesu@@',
 'cristo',
 ',',
 'el',
 'hijo',
 'de']

In [66]:
len(spa_bible_tokenized)

38497

In [67]:
spa_bible_tokenized_types = Counter(spa_bible_tokenized)
len(spa_bible_tokenized_types)

2297

In [68]:
spa_bible_tokenized_types.most_common(40)

[(',', 1946),
 ('y', 1209),
 ('.', 1099),
 ('de', 1038),
 ('a', 1000),
 ('que', 929),
 ('los', 736),
 ('la', 629),
 ('en', 586),
 ('el', 583),
 (':', 539),
 ('se', 526),
 ('«', 423),
 ('»', 423),
 ('jesús', 422),
 ('lo', 414),
 ('es', 386),
 ('no', 322),
 ('le', 320),
 ('les', 290),
 ('dijo', 258),
 ('ó', 221),
 ('con', 220),
 ('pero', 217),
 ('al', 216),
 ('¿', 196),
 ('?', 195),
 ('por', 194),
 ('o', 192),
 ('para', 175),
 ('su', 171),
 ('del', 165),
 ('un', 159),
 ('aron', 158),
 ('¡', 158),
 ('!', 158),
 ('an', 154),
 ('cuando', 148),
 ('ed@@', 147),
 ('las', 145)]

In [69]:
rprint("Biblia español")
rprint(f"Tipos ([bright_magenta]word-base[/]): {len(spa_bible_types)}")
rprint(f"Tipos ([bright_green]sub-word[/]): {len(spa_bible_tokenized_types)}")

Biblia español

Tipos (word-base): 3317

Tipos (sub-word): 2297

#### OOV: out of vocabulary

Palabras que se vieron en el entrenamiento pero no estan en el test

In [70]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    spa_bible_words, test_size=0.3, random_state=42
)
rprint(len(train_data), len(test_data))

21051 9022

In [71]:
s_1 = {"a", "b", "c", "d", "e"}
s_2 = {"a", "x", "y", "d"}
rprint(s_1 - s_2)
rprint(s_2 - s_1)

{'e', 'b', 'c'}

{'y', 'x'}

In [72]:
oov_test = set(test_data) - set(train_data)

In [73]:
for word in list(oov_test)[:3]:
    rprint(f"{word} in train: {word in set(train_data)}")

harina in train: False

instrucciones in train: False

confirmando in train: False

In [74]:
train_tokenized, test_tokenized = train_test_split(
    spa_bible_tokenized, test_size=0.3, random_state=42
)
rprint(len(train_tokenized), len(test_tokenized))

26947 11550

In [75]:
oov_tokenized_test = set(test_tokenized) - set(train_tokenized)

In [78]:
rprint("OOV ([bright_magenta]word-base[/]):", len(oov_test))
rprint("OOV ([bright_green]sub-word[/]):", len(oov_tokenized_test))

OOV (word-base): 533

OOV (sub-word): 216

### Type-token Ratio (TTR)

- Una forma de medir la variación del vocabulario en un corpus
- Este se calcula como $TTR = \frac{len(types)}{len(tokens)}$
- Puede ser útil para monitorear la variación lexica de un texto

In [90]:
stemmer = SnowballStemmer("spanish")
spa_bible_stemmed = [stemmer.stem(word) for word in spa_bible_words]
spa_bible_stemmed_types = set(spa_bible_stemmed)

In [ ]:
rprint("Bible Spanish Information")
rprint("Tokens:", len(spa_bible_words))
rprint("Types ([bright_magenta]word-base[/]):", len(spa_bible_types))
rprint("Types ([bright_yellow]stemmed[/])", len(spa_bible_stemmed_types))
rprint("Types ([bright_green]BPE[/]):", len(spa_bible_tokenized_types))
rprint(
    "TTR ([bright_magenta]word-base[/]):", len(spa_bible_types) / len(spa_bible_words)
)
rprint(
    "TTR ([bright_yellow]stemmed[/]):",
    len(spa_bible_stemmed_types) / len(spa_bible_stemmed),
)
rprint(
    "TTR ([bright_green]BPE[/]):",
    len(spa_bible_tokenized_types) / len(spa_bible_tokenized),
)

Bible Spanish Information

Tokens: 30073

Types (word-base): 3317

Types (stemmed) 1897

Types (BPE): 2297

TTR (word-base): 0.11029827419944802

TTR (stemmed): 0.0630798390582915

TTR (BPE): 0.05966698703795101

## Word Embeddings (W2V, GloVe) 

Vamos a entrenar nuestras propias representaciones vectoriales

![we](https://miro.medium.com/v2/resize:fit:2000/1*SYiW1MUZul1NvL1kc1RxwQ.png)

### Datos: Noticias en Español

In [7]:
news_databuilder = load_dataset_builder("LeoCordoba/CC-NEWS-ES", "mx")

In [8]:
rprint(news_databuilder.info)

DatasetInfo(
    description='',
    citation=' ',
    homepage='',
    license='',
    features={
        'country': Value(dtype='string', id=None),
        'text': Value(dtype='string', id=None),
        'id': Value(dtype='int32', id=None)
    },
    post_processed=None,
    supervised_keys=None,
    builder_name='parquet',
    dataset_name='cc-news-es',
    config_name='mx',
    version=0.0.0,
    splits={
        'train': SplitInfo(
            name='train',
            num_bytes=919134987,
            num_examples=652418,
            shard_lengths=[354000, 298418],
            dataset_name='cc-news-es'
        )
    },
    download_checksums={
        'hf://datasets/LeoCordoba/CC-NEWS-ES@b59e79414d484cb490a723aff49fc1dc4a977367/mx/train/0000.parquet': {
            'num_bytes': 280963807,
            'checksum': None
        },
        'hf://datasets/LeoCordoba/CC-NEWS-ES@b59e79414d484cb490a723aff49fc1dc4a977367/mx/train/0001.parquet': {
            'num_bytes': 235933763,
            'checksum': None
        }
    },
    download_size=516897570,
    post_processing_size=None,
    dataset_size=919134987,
    size_in_bytes=1436032557
)

In [ ]:
news_dataset = load_dataset(
    "LeoCordoba/CC-NEWS-ES", "mx", split="train", streaming=True
)

In [10]:
for post in news_dataset.take(1):
    rprint(post["text"])

La procuradora general de Justicia, Ernestina Godoy, informó que se logró imputar el delito de violación agravada a
cuatro policías preventivos que habrían abusado sexualmente de una menor de edad a bordo de una patrulla en calles 
de la demarcación Azcapotzalco; sin embargo, el secretario de Seguridad Ciudadana, Jesús Orta Martínez, dijo que la
dependencia a su cargo aún no ha sido notificada.    Por la tarde, el jefe de la policía capitalina, luego de un 
encuentro con el presidente de la Cámara Nacional de Comercio, Servicios y Turismo de la Ciudad de México, Nathan 
Poplawsky, comentó que los uniformados siguen en resguardo por la Dirección General de Asuntos Internos y no han 
sido suspendidos.

In [11]:
from gensim.utils import simple_preprocess

rprint(simple_preprocess(post["text"], deacc=True)[:10])

['la', 'procuradora', 'general', 'de', 'justicia', 'ernestina', 'godoy', 'informo', 'que', 'se']

In [ ]:
from datasets.dataset_dict import IterableDatasetDict
from datasets.iterable_dataset import IterableDataset
from tqdm.notebook import tqdm
from datasets import load_dataset
from gensim.utils import simple_preprocess


class CCNewsExtractor:
    """
    Iterador optimizado para CC-NEWS-ES + Word2Vec.
    Diseñado para alta velocidad y compatibilidad con los epochs de Gensim.
    """

    def __init__(self, lang: str = "mx", max_posts: int = -1):
        self.dataset = load_dataset(
            "LeoCordoba/CC-NEWS-ES", name=lang, split="train", streaming=True
        )
        self.max_posts = max_posts

        # Precompilar la expresión regular es considerablemente más rápido
        # en bucles anidados que inicializarla en cada pasada.
        self.sent_splitter = re.compile(r"[.!?\n]+")

    def __iter__(self):
        for item in tqdm(self.dataset.take(self.max_posts)):
            text = item.get("text", "")
            if not text:
                continue

            words = simple_preprocess(text, deacc=False, min_len=2)

            if not words:
                continue
            yield words

In [13]:
# Uso con tu función train_model
iterator = CCNewsExtractor(lang="mx", max_posts=3)

In [14]:
for i in iterator:
    rprint(i[:10])

0it [00:00, ?it/s]

['la', 'procuradora', 'general', 'de', 'justicia', 'ernestina', 'godoy', 'informó', 'que', 'se']

['en', 'la', 'réplica', 'el', 'gobernador', 'con', 'licencia', 'espetó', 'quiero', 'precisarle']

['la', 'estadunidense', 'claire', 'rasmus', 'firmó', 'el', 'primero', 'de', 'los', 'siete']

In [ ]:
%%time
sentences = CCNewsExtractor(lang="mx", max_posts=10)

CPU times: user 22.6 ms, sys: 7.2 ms, total: 29.8 ms
Wall time: 2.29 s


In [18]:
for sentence in sentences:
    print(sentence)

0it [00:00, ?it/s]

['la', 'procuradora', 'general', 'de', 'justicia', 'ernestina', 'godoy', 'informó', 'que', 'se', 'logró', 'imputar', 'el', 'delito', 'de', 'violación', 'agravada', 'cuatro', 'policías', 'preventivos', 'que', 'habrían', 'abusado', 'sexualmente', 'de', 'una', 'menor', 'de', 'edad', 'bordo', 'de', 'una', 'patrulla', 'en', 'calles', 'de', 'la', 'demarcación', 'azcapotzalco', 'sin', 'embargo', 'el', 'secretario', 'de', 'seguridad', 'ciudadana', 'jesús', 'orta', 'martínez', 'dijo', 'que', 'la', 'dependencia', 'su', 'cargo', 'aún', 'no', 'ha', 'sido', 'notificada', 'por', 'la', 'tarde', 'el', 'jefe', 'de', 'la', 'policía', 'capitalina', 'luego', 'de', 'un', 'encuentro', 'con', 'el', 'presidente', 'de', 'la', 'cámara', 'nacional', 'de', 'comercio', 'servicios', 'turismo', 'de', 'la', 'ciudad', 'de', 'méxico', 'nathan', 'poplawsky', 'comentó', 'que', 'los', 'uniformados', 'siguen', 'en', 'resguardo', 'por', 'la', 'dirección', 'general', 'de', 'asuntos', 'internos', 'no', 'han', 'sido', 'suspend

In [19]:
from gensim.models import word2vec

In [28]:
EMB_MODELS_DIR = os.path.join(MODELS_PATH, "embeddings")

os.makedirs(EMB_MODELS_DIR, exist_ok=True)

In [ ]:
from enum import Enum


class Algorithms(Enum):
    CBOW = 0
    SKIP_GRAM = 1

In [31]:
def load_model(model_path: str):
    try:
        print(model_path)
        return word2vec.Word2Vec.load(model_path)
    except FileNotFoundError:
        print(f"[WARN] Model not found in path {model_path}")
        return None

In [ ]:
def train_model(
    sentences: list,
    model_name: str,
    vector_size: int,
    window=5,
    workers=2,
    algorithm=Algorithms.CBOW,
):
    model_name_params = f"{model_name}-vs{vector_size}-w{window}-{algorithm.name}.model"
    model_path = os.path.join(EMB_MODELS_DIR, model_name_params)
    if load_model(model_path) is not None:
        print(f"Already exists the model {model_path}")
        return load_model(model_path)
    print(f"TRAINING: {model_path}")
    if algorithm in [Algorithms.CBOW, Algorithms.SKIP_GRAM]:
        model = word2vec.Word2Vec(
            sentences,
            vector_size=vector_size,
            window=window,
            workers=workers,
            sg=algorithm.value,
            seed=42,
        )
    else:
        print("[ERROR] algorithm not implemented yet :p")
        return
    try:
        model.save(model_path)
    except:
        print(f"[ERROR] Saving model at {model_path}")
    return model

### CBOW

In [ ]:
skipm_gram_model = load_model(
    os.path.join(EMB_MODELS_DIR, "eswiki-xl-300-SKIP_GRAM.model")
)

./models/embeddings/eswiki-xl-300-SKIP_GRAM.model
[WARN] Model not found in path ./models/embeddings/eswiki-xl-300-SKIP_GRAM.model


In [ ]:
%%time
cbow_model = train_model(
    CCNewsExtractor(lang="mx", max_posts=100_000),
    "eswiki",
    vector_size=100,
    window=3,
    workers=6,
    algorithm=Algorithms.CBOW,
)

./models/embeddings/eswiki-vs100-w3-CBOW.model
[WARN] Model not found in path ./models/embeddings/eswiki-vs100-w3-CBOW.model
TRAINING: ./models/embeddings/eswiki-vs100-w3-CBOW.model


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

CPU times: user 7min 5s, sys: 16.9 s, total: 7min 21s
Wall time: 6min 34s


### Skip gram

In [ ]:
%%time
skip_gram_model = train_model(
    CCNewsExtractor(lang="mx", max_posts=100_000),
    "es_news_hf",
    300,
    5,
    workers=12,
    algorithm=Algorithms.SKIP_GRAM,
)

./models/embeddings/es_news_hf-vs300-w5-SKIP_GRAM.model
[WARN] Model not found in path ./models/embeddings/es_news_hf-vs300-w5-SKIP_GRAM.model
TRAINING: ./models/embeddings/es_news_hf-vs300-w5-SKIP_GRAM.model


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

CPU times: user 21min 50s, sys: 22.1 s, total: 22min 12s
Wall time: 9min 3s


In [ ]:
def report_stats(model) -> None:
    """Print report of a model"""
    print(
        "Number of words in the corpus used for training the model: ",
        model.corpus_count,
    )
    print("Number of words in the model: ", len(model.wv.index_to_key))
    print("Time [s], required for training the model: ", model.total_train_time)
    print("Count of trainings performed to generate this model: ", model.train_count)
    print("Length of the word2vec vectors: ", model.vector_size)
    print("Applied context length for generating the model: ", model.window)

In [36]:
report_stats(cbow_model)

Number of words in the corpus used for training the model:  100000
Number of words in the model:  72998
Time [s], required for training the model:  338.614065972999
Count of trainings performed to generate this model:  1
Length of the word2vec vectors:  100
Applied context length for generating the model:  3


In [58]:
report_stats(skip_gram_model)

Number of words in the corpus used for training the model:  100000
Number of words in the model:  72998
Time [s], required for training the model:  437.1355642360013
Count of trainings performed to generate this model:  1
Length of the word2vec vectors:  300
Applied context length for generating the model:  5


### Operaciones con los vectores entrenados

Veremos operaciones comunes sobre vectores. Estos resultados dependeran del modelo que hayamos cargado en memoria

In [59]:
models = {
    Algorithms.CBOW: cbow_model,
    Algorithms.SKIP_GRAM: skip_gram_model,
}

In [60]:
model = models[Algorithms.SKIP_GRAM]

In [61]:
for index, word in enumerate(model.wv.index_to_key):
    if index == 100:
        break
    print(f"word #{index}/{len(model.wv.index_to_key)} is {word}")

word #0/72998 is de
word #1/72998 is la
word #2/72998 is que
word #3/72998 is el
word #4/72998 is en
word #5/72998 is los
word #6/72998 is del
word #7/72998 is se
word #8/72998 is con
word #9/72998 is por
word #10/72998 is las
word #11/72998 is un
word #12/72998 is para
word #13/72998 is una
word #14/72998 is no
word #15/72998 is su
word #16/72998 is al
word #17/72998 is es
word #18/72998 is lo
word #19/72998 is como
word #20/72998 is más
word #21/72998 is sus
word #22/72998 is este
word #23/72998 is pero
word #24/72998 is fue
word #25/72998 is ha
word #26/72998 is sin
word #27/72998 is méxico
word #28/72998 is ya
word #29/72998 is le
word #30/72998 is años
word #31/72998 is esta
word #32/72998 is personas
word #33/72998 is entre
word #34/72998 is también
word #35/72998 is gobierno
word #36/72998 is mil
word #37/72998 is sobre
word #38/72998 is estado
word #39/72998 is son
word #40/72998 is si
word #41/72998 is todos
word #42/72998 is está
word #43/72998 is ser
word #44/72998 is donde


In [62]:
gato_vec = model.wv["gato"]
print(gato_vec[:10])
print(len(gato_vec))

[ 0.18492897 -0.03264451 -0.15665221 -0.319803   -0.02698543  0.08257209
 -0.15770938 -0.17113765  0.11688305  0.36170018]
300


In [63]:
try:
    agustisidad_vec = model.wv["agusticidad"]
except KeyError:
    print("OOV founded!")


OOV founded!


In [ ]:
agustisidad_vec[:10]
len(agustisidad_vec)

In [76]:
model.wv.most_similar("mercado", topn=5)

[('accionario', 0.5775202512741089),
 ('mercados', 0.5415079593658447),
 ('cambiario', 0.5390856862068176),
 ('bursátil', 0.47074949741363525),
 ('líquidas', 0.4693392813205719)]

Podemos ver como la similitud entre palabras decrece

In [77]:
word_pairs = [
    ("automóvil", "camión"),
    ("automóvil", "bicicleta"),
    ("automóvil", "cereal"),
    ("automóvil", "conde"),
]

for w1, w2 in word_pairs:
    print(f"{w1} - {w2} {model.wv.similarity(w1, w2)}")

automóvil - camión 0.590190589427948
automóvil - bicicleta 0.42268607020378113
automóvil - cereal 0.22943748533725739
automóvil - conde 0.14540159702301025


In [ ]:
# rey es a hombre como ___ a mujer
# londres es a inglaterra como ____ a vino
model.wv.most_similar(positive=["saltillo", "morelos"], negative=["cuernavaca"])

[('coahuila', 0.613200306892395),
 ('torreón', 0.48774412274360657),
 ('arteaga', 0.47748732566833496),
 ('durango', 0.46789276599884033),
 ('múzquiz', 0.45535987615585327),
 ('monclova', 0.44698217511177063),
 ('arizpe', 0.44026798009872437),
 ('hgs', 0.4273076057434082),
 ('parras', 0.42663225531578064),
 ('laguna', 0.414446085691452)]

In [82]:
model.wv.doesnt_match(["disco", "música", "mantequilla", "cantante"])

'mantequilla'

In [55]:
model.wv.similarity("noche", "noches")

np.float32(0.38414505)

#### Evaluación

`Word2Vec` es una tarea de entrenamiento semi-supervisada, por lo tanto, es difícil evaluar el desempeño de un modelo. La evaluación dependerá de la tarea.

Sin embargo, Google liberó un conjunto de evaluación con ejemplos semánticos/sintácticos. Se sigue la forma "A es a B como C es a D". Por ejemplo, "tokio es a japon como berlin es a alemania".

Se tienen varias categorias como comparaciones sintácticas, capitales, miembros de una familia, etc.

In [81]:
from gensim.test.utils import datapath

model.wv.evaluate_word_analogies(datapath("questions-words.txt"))

(0.03231597845601436,
 [{'section': 'capital-common-countries',
   'correct': [('CANBERRA', 'AUSTRALIA', 'BEIJING', 'CHINA'),
    ('HANOI', 'VIETNAM', 'BEIJING', 'CHINA')],
   'incorrect': [('BEIJING', 'CHINA', 'CANBERRA', 'AUSTRALIA'),
    ('BEIJING', 'CHINA', 'HANOI', 'VIETNAM'),
    ('BEIJING', 'CHINA', 'HAVANA', 'CUBA'),
    ('BEIJING', 'CHINA', 'LONDON', 'ENGLAND'),
    ('BEIJING', 'CHINA', 'MADRID', 'SPAIN'),
    ('BEIJING', 'CHINA', 'OTTAWA', 'CANADA'),
    ('BEIJING', 'CHINA', 'PARIS', 'FRANCE'),
    ('BEIJING', 'CHINA', 'TOKYO', 'JAPAN'),
    ('CANBERRA', 'AUSTRALIA', 'HANOI', 'VIETNAM'),
    ('CANBERRA', 'AUSTRALIA', 'HAVANA', 'CUBA'),
    ('CANBERRA', 'AUSTRALIA', 'LONDON', 'ENGLAND'),
    ('CANBERRA', 'AUSTRALIA', 'MADRID', 'SPAIN'),
    ('CANBERRA', 'AUSTRALIA', 'OTTAWA', 'CANADA'),
    ('CANBERRA', 'AUSTRALIA', 'PARIS', 'FRANCE'),
    ('CANBERRA', 'AUSTRALIA', 'TOKYO', 'JAPAN'),
    ('HANOI', 'VIETNAM', 'HAVANA', 'CUBA'),
    ('HANOI', 'VIETNAM', 'LONDON', 'ENGLAND'),
   

## Modelos del Lenguaje Neuronales (Bengio)

- [(Bengio et al 2003)](https://dl.acm.org/doi/10.5555/944919.944966) proponen una arquitecura neuronal como alternativa a los modelos del lenguaje estadísticos
- Esta arquitectura lidia mejor con los casos donde las probabilidades se hacen cero, sin necesidad de aplicar una técnica de smoothing.

<p float="left">
  <img src="https://toppng.com/public/uploads/preview/at-the-movies-will-smith-meme-tada-11562851401lnexjqtwf9.png" width="100" />
  <img src="https://abhinavcreed13.github.io/assets/images/bengio-model.png" width="600"/>
</p>

In [86]:
def lm_preprocess_corpus(corpus: list[str]) -> list[str]:
    """Función de preprocesamiento para LM

    Esta función está diseñada para preprocesar
    corpus para modelos del lenguaje neuronales.
    Agrega tokens de inicio y fin, normaliza
    palabras a minusculas
    """
    preprocessed_corpus = []
    for sent in corpus:
        result = [word.lower() for word in sent]
        # Al final de la oración
        result.append("<EOS>")
        result.insert(0, "<BOS>")
        preprocessed_corpus.append(result)
    return preprocessed_corpus

In [87]:
def get_words_freqs(corpus: list[list[str]]):
    """Calcula la frecuencia de las palabras en un corpus"""
    words_freqs = {}
    for sentence in corpus:
        for word in sentence:
            words_freqs[word] = words_freqs.get(word, 0) + 1
    return words_freqs

In [ ]:
UNK_LABEL = "<UNK>"


def get_words_indexes(words_freqs: dict) -> dict:
    """Calcula los indices de las palabras dadas sus frecuencias"""
    result = {}
    for idx, word in enumerate(words_freqs.keys()):
        # Happax legomena happends
        if words_freqs[word] == 1:
            # Temp index for unknowns
            result[UNK_LABEL] = len(words_freqs)
        else:
            result[word] = idx

    return {word: idx for idx, word in enumerate(result.keys())}, {
        idx: word for idx, word in enumerate(result.keys())
    }

In [153]:
import nltk

nltk.download("gutenberg")
nltk.download("abc")
nltk.download("genesis")
nltk.download("inaugural")
nltk.download("state_union")
nltk.download("webtext")
nltk.download("punkt_tab")

[nltk_data] Downloading package gutenberg to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package abc to /home/umoqnier/nltk_data...
[nltk_data]   Package abc is already up-to-date!
[nltk_data] Downloading package genesis to /home/umoqnier/nltk_data...
[nltk_data]   Package genesis is already up-to-date!
[nltk_data] Downloading package inaugural to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!
[nltk_data] Downloading package state_union to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package state_union is already up-to-date!
[nltk_data] Downloading package webtext to /home/umoqnier/nltk_data...
[nltk_data]   Package webtext is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [156]:
from nltk.corpus import abc, genesis, gutenberg, inaugural, state_union, webtext

# Exploración del corpus
total_sents = 0
corpora = []

plaintext_corpora = {
    "abc": abc,
    "Gutenberg": gutenberg,
    "Genesis": genesis,
    "Inaugural": inaugural,
    "State Union": state_union,
    "Web": webtext,
}

for title, corpus in plaintext_corpora.items():
    corpus_sents = lm_preprocess_corpus(corpus.sents())
    corpus_len = len(corpus_sents)
    total_sents += corpus_len
    print(f"{title} sents={corpus_len}")
    corpora.extend(corpus_sents)
print(f"Total={total_sents}")

abc sents=29059
Gutenberg sents=98552
Genesis sents=13640
Inaugural sents=5395
State Union sents=17930
Web sents=25733
Total=190309


In [157]:
len(corpora)

190309

In [159]:
corpora[42]

['<BOS>',
 'awi',
 'is',
 'showing',
 'wool',
 'blend',
 't',
 '-',
 'shirts',
 'and',
 'casual',
 'wear',
 'to',
 'manufacturers',
 'this',
 'week',
 'at',
 'the',
 'largest',
 'trade',
 'show',
 'for',
 'the',
 'industry',
 'being',
 'held',
 'in',
 'germany',
 '.',
 '<EOS>']

In [161]:
words_freqs = get_words_freqs(corpora)

In [162]:
words_freqs["the"]

219272

In [163]:
len(words_freqs)

85406

In [164]:
count = 0
for word, freq in words_freqs.items():
    if freq == 1 and count <= 10:
        print(word, freq)
        count += 1

totalled 1
skimmed 1
ploy 1
multinationals 1
truckloads 1
conner 1
9m 1
elevators 1
devonport 1
macleod 1
stopwork 1


In [165]:
words_indexes, index_to_word = get_words_indexes(words_freqs)

In [166]:
words_indexes["god"]

9573

In [168]:
index_to_word[9573]

'god'

In [169]:
len(words_indexes)

50998

In [170]:
len(index_to_word)

50998

In [171]:
def get_word_id(words_indexes: dict, word: str) -> int:
    """Obtiene el id de una palabra dada

    Si no se encuentra la palabra se regresa el id
    del token UNK
    """
    unk_word_id = words_indexes[UNK_LABEL]
    return words_indexes.get(word, unk_word_id)

### Obtenemos trigramas

Convertiremos los trigramas obtenidos a secuencias de idx, y preparamos el conjunto de entrenamiento $x$ y $y$

- x: Contexto
- y: Predicción de la siguiente palabra

In [ ]:
from nltk import ngrams


def get_train_test_data(
    corpus: list[list[str]], words_indexes: dict, n: int
) -> tuple[list, list]:
    """Obtiene el conjunto de train y test

    Requerido en el step de entrenamiento del modelo neuronal
    """
    x_train = []
    y_train = []
    for sent in corpus:
        n_grams = ngrams(sent, n)
        for w1, w2, w3 in n_grams:
            x_train.append(
                [get_word_id(words_indexes, w1), get_word_id(words_indexes, w2)]
            )
            y_train.append([get_word_id(words_indexes, w3)])
    return x_train, y_train

### Preparando Pytorch

$x' = e(x_1) \oplus e(x_2)$

$h = \tanh(W_1 x' + b)$

$y = softmax(W_2 h)$

In [173]:
# cargamos bibliotecas
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import time

In [174]:
# Setup de parametros
EMBEDDING_DIM = 200
CONTEXT_SIZE = 2
BATCH_SIZE = 256
H = 100
torch.manual_seed(42)
# Tamaño del Vocabulario
V = len(words_indexes)

In [176]:
x_train, y_train = get_train_test_data(corpora, words_indexes, n=3)

In [ ]:
import numpy as np

train_set = np.concatenate((x_train, y_train), axis=1)
# partimos los datos de entrada en batches
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE)

### Creamos la arquitectura del modelo

In [ ]:
# Trigram Neural Network Model
class TrigramModel(nn.Module):
    """Clase padre: https://pytorch.org/docs/stable/generated/torch.nn.Module.html"""

    def __init__(self, vocab_size, embedding_dim, context_size, h):
        super(TrigramModel, self).__init__()
        self.context_size = context_size
        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, h)
        self.linear2 = nn.Linear(h, vocab_size)

    def forward(self, inputs):
        # x': concatenation of x1 and x2 embeddings   -->
        # self.embeddings regresa un vector por cada uno de los índices que se les pase como entrada.
        # view() les cambia el tamaño para concatenarlos
        embeds = self.embeddings(inputs).view(
            (-1, self.context_size * self.embedding_dim)
        )
        # h: tanh(W_1.x' + b)  -->
        out = torch.tanh(self.linear1(embeds))
        # W_2.h                 -->
        out = self.linear2(out)
        # log_softmax(W_2.h)      -->
        # dim=1 para que opere sobre renglones, pues al usar batchs tenemos varios vectores de salida
        log_probs = F.log_softmax(out, dim=1)

        return log_probs

In [ ]:
# Seleccionar la GPU si está disponible
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)

In [180]:
NN_MODELS_PATH = os.path.join(MODELS_PATH, "nn")

os.makedirs(NN_MODELS_PATH, exist_ok=True)

LM_PATH = os.path.join(NN_MODELS_PATH, "trigrams_nlm_cpu_epoch3.pt")

In [183]:
torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device {device}")

# 1. Pérdida. Negative log-likelihood loss
loss_function = nn.NLLLoss()

# 2. Instanciar el modelo y enviarlo a device
model = TrigramModel(V, EMBEDDING_DIM, CONTEXT_SIZE, H).to(device)

# 3. Optimización. ADAM optimizer
optimizer = optim.Adam(model.parameters(), lr=2e-3)

# ------------------------- TRAIN & SAVE MODEL ------------------------
EPOCHS = 1
for epoch in range(EPOCHS):
    st = time.time()
    print("\n--- Training model Epoch: {} ---".format(epoch))
    for it, data_tensor in enumerate(train_loader):
        # Mover los datos al dispositivo
        context_tensor = data_tensor[:, 0:2].to(device)
        target_tensor = data_tensor[:, 2].to(device)

        model.zero_grad()

        # FORWARD:
        log_probs = model(context_tensor)

        # compute loss function
        loss = loss_function(log_probs, target_tensor)

        # BACKWARD:
        loss.backward()
        optimizer.step()

        if it % 500 == 0:
            print(
                "Training Iteration {} of epoch {} complete. Loss: {}; Time taken (s): {}".format(
                    it, epoch, loss.item(), (time.time() - st)
                )
            )
            st = time.time()

    # saving model
    model_path = os.path.join(
        NN_MODELS_PATH, f"lm_large_{device}_context_{CONTEXT_SIZE}_epoch_{epoch}.dat"
    )
    torch.save(model.state_dict(), model_path)
    print(f"Model saved for epoch={epoch} at {model_path}")

Training on device cpu

--- Training model Epoch: 0 ---
Training Iteration 0 of epoch 0 complete. Loss: 10.874826431274414; Time taken (s): 0.2959465980529785
Training Iteration 500 of epoch 0 complete. Loss: 6.69818639755249; Time taken (s): 84.89241743087769
Training Iteration 1000 of epoch 0 complete. Loss: 5.260696887969971; Time taken (s): 75.46042060852051
Training Iteration 1500 of epoch 0 complete. Loss: 6.446754455566406; Time taken (s): 74.41151881217957
Training Iteration 2000 of epoch 0 complete. Loss: 6.269434452056885; Time taken (s): 74.40606999397278
Training Iteration 2500 of epoch 0 complete. Loss: 5.515120983123779; Time taken (s): 75.04196500778198
Training Iteration 3000 of epoch 0 complete. Loss: 6.860296726226807; Time taken (s): 83.3623263835907


KeyboardInterrupt: 

In [126]:
model

TrigramModel(
  (embeddings): Embedding(12476, 200)
  (linear1): Linear(in_features=400, out_features=100, bias=True)
  (linear2): Linear(in_features=100, out_features=12476, bias=True)
)

In [127]:
def get_model(path: str) -> TrigramModel:
    """Obtiene modelo de pytorch desde disco"""
    model_loaded = TrigramModel(V, EMBEDDING_DIM, CONTEXT_SIZE, H)
    model_loaded.load_state_dict(torch.load(path))
    model_loaded.eval()
    return model_loaded

In [ ]:
# model = get_model(PATH)
W1 = "<BOS>"
W2 = "my"

IDX1 = get_word_id(words_indexes, W1)
IDX2 = get_word_id(words_indexes, W2)

# Obtenemos Log probabidades p(W3|W2,W1)
probs = model(torch.tensor([[IDX1, IDX2]]).to(device)).detach().tolist()

In [132]:
len(probs[0])

12476

In [190]:
# Creamos diccionario con {idx: logprob}
model_probs = {}
for idx, p in enumerate(probs[0]):
    model_probs[idx] = p

# Sort:
model_probs_sorted = sorted(
    ((prob, idx) for idx, prob in model_probs.items()), reverse=True
)

# Printing word  and prob (retrieving the idx):
topcandidates = 0
for prob, idx in model_probs_sorted:
    # Retrieve the word associated with that idx
    word = index_to_word[idx]
    print(idx, word, prob, np.exp(prob))

    topcandidates += 1

    if topcandidates > 10:
        break

279 few -3.6791160106658936 0.025245281533945257
172 south -3.6832587718963623 0.025140912697208292
8929 distort -3.8009421825408936 0.022349704431734622
9069 missile -3.826054334640503 0.021795443721321375
438 germany -4.23629093170166 0.014461129811886788
383 wine -4.960487365722656 0.007009510804683132
129 australia -5.03106689453125 0.0065318380694335005
458 particularly -5.10292911529541 0.006078914620559122
214 d -5.138623237609863 0.005865759914325895
818 liberals -5.193914413452148 0.00555023830974884
9025 nearest -5.232447624206543 0.005340437907585


In [134]:
print(index_to_word.get(model_probs_sorted[0][1]))

father


### Generacion de lenguaje

In [214]:
def get_likely_words(
    model: TrigramModel,
    context: str,
    words_indexes: dict,
    index_to_word: dict,
) -> list[tuple]:
    model_probs = {}
    words = context.split()
    idx_word_1 = get_word_id(words_indexes, words[0])
    idx_word_2 = get_word_id(words_indexes, words[1])
    probs = model(torch.tensor([[idx_word_1, idx_word_2]]).to(device)).detach().tolist()

    for idx, p in enumerate(probs[0]):
        model_probs[idx] = p

    # Strategy: Sort and get top-K words to generate text
    return sorted(
        ((prob, index_to_word[idx]) for idx, prob in model_probs.items()), reverse=True
    )

In [215]:
sentence = "this is"
get_likely_words(model, sentence, words_indexes, index_to_word)[:3]

[(-1.7313828468322754, 'the'),
 (-2.104790210723877, 'a'),
 (-2.7494850158691406, 'not')]

In [216]:
import random
from random import randint

def get_next_top_p_word(words: list[tuple[float, str]], p: float = 0.8) -> str:
    """
    Selecciona la siguiente palabra utilizando Nucleus Sampling (Top-p).
    
    Params:
    - words: Lista de tuplas (palabra, probabilidad).
    - p: Umbral de masa de probabilidad acumulada (típicamente entre 0.8 y 0.95).
    """
    if not words:
        return "<EOS>"
        
    # Aseguramos que la lista esté ordenada de mayor a menor probabilidad
    # sorted_words = sorted(words, key=lambda x: x[1], reverse=True)
    
    valid_words = []
    valid_probs = []
    cumulative_prob = 0.0
    
    # Recolectamos palabras hasta que la suma de probabilidades alcance el umbral 'p'
    for log_prob, word in words:
        prob = np.exp(log_prob)
        valid_words.append(word)
        valid_probs.append(prob)
        cumulative_prob += prob
        
        if cumulative_prob >= p:
            break
            
    # Muestreamos una palabra del subconjunto válido (núcleo) usando sus probabilidades como pesos.
    # random.choices devuelve una lista, por lo que extraemos el elemento [0]
    return random.choices(valid_words, weights=valid_probs, k=1)[0]


def get_next_word(words: list[tuple[float, str]]) -> str:
    # From a top-K list of words get a random word
    return words[randint(0, len(words) - 1)][1]

In [218]:
get_next_top_p_word(get_likely_words(model, sentence, words_indexes, index_to_word))

'always'

In [221]:
MAX_TOKENS = 50
TOP_P = 0.7


def generate_text(
    model: TrigramModel,
    history: str,
    words_indexes: dict,
    index_to_word: dict,
    tokens_count: int = 0,
) -> None:
    next_word = get_next_top_p_word(
        get_likely_words(
            model, history, words_indexes, index_to_word
        ), p=TOP_P
    )
    print(next_word, end=" ")
    tokens_count += 1
    if tokens_count == MAX_TOKENS or next_word == "<EOS>":
        return
    generate_text(
        model,
        history.split()[1] + " " + next_word,
        words_indexes,
        index_to_word,
        tokens_count,
    )

In [232]:
sentence = "god said"
print(sentence, end=" ")
generate_text(model, sentence, words_indexes, index_to_word)

god said that it will be very well to leave . <EOS> 

# Práctica 4: Evaluación de modelos del lenguaje

**Fecha: 5 de Mayo 2026 11:59pm**

La calidad de un modelo del lenguaje puede ser evaluado por medio de su perplejidad (perplexity)

- Investigar como calcular la perplejidad de un modelo del lenguaje y como evaluarlo con esa medida
    - Incluir en el `README.md` de su entrega una síntesis de esta investigación (Un par de parrafos)
- Evalua el modelo entrenado en clase con los corpus de `nltk`
- Entrena un nuevo modelo del lenguaje neuronal con los corpus de `nltk` aplicando previamente sub-word tokenization al corpus 
    - Puedes utilizar un modelo de tokenizacion pre-entrenado o entrenar uno desde cero
    - TODO: Test de evauación
- Evalua tu modelo calculando su perplejidad
    - Compara los resultados de la evaluación de los ambos modelos.
    - ¿Cúal es mejor? ¿Por qué?

## EXTRA

- Diseña una estrategia de generación de usando el modelo del lenguaje entrenado con sub-word tokenization
- Se deben generar secuencias de palabras (no subwords)